# Predictive Analysis: Explainable Model Selection

This notebook predicts `log_box_office` on unseen movies using only explainable regression models.

The removed interaction feature is not part of the dataset or candidate pool. The workflow compares simple baselines, the inferential model, the full candidate-pool model, best-subset selection, and forward-stepwise selection.

## Prerequisites and Modelling Rules

Before prediction, this notebook checks that:

- `data/final/final.csv` exists
- the removed interaction column is absent
- leakage columns such as raw box-office revenue are not used as predictors
- all model candidates are numeric and available
- model selection uses cross-validation on the training set only

Only explainable linear regression models are used. This keeps the predictive workflow aligned with the assignment and makes feature effects defensible.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)
warnings.filterwarnings('ignore', category=RuntimeWarning)


def resolve_data_path() -> Path:
    for base in [Path.cwd(), Path.cwd().parent]:
        candidate = base / 'data' / 'final' / 'final.csv'
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find data/final/final.csv from the current working directory.')

import itertools
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

TARGET = 'log_box_office'
TRAIN_SIZE = 0.80
TEST_SIZE = 0.20
RANDOM_STATE = 42
N_SPLITS = 5

CANDIDATE_PREDICTORS = [
    'audienceScore',
    'tomatoMeter',
    'runtimeMinutes',
    'initial_top_critic_review_count',
    'initial_positive_review_ratio',
    'initial_combined_sentiment_score',
    'log_initial_review_count',
]

INFERENTIAL_MODEL_PREDICTORS = [
    'audienceScore',
    'initial_combined_sentiment_score',
    'log_initial_review_count',
]

LEAKAGE_COLUMNS = ['boxOffice', 'box_office_num']
REQUIRED_COLUMNS = ['id', 'title', 'box_office_num', TARGET, *CANDIDATE_PREDICTORS]

DATA_PATH = resolve_data_path()
df = pd.read_csv(DATA_PATH)

missing_required = [column for column in REQUIRED_COLUMNS if column not in df.columns]
if missing_required:
    raise KeyError(f'Missing required columns: {missing_required}')

leakage_used = [column for column in LEAKAGE_COLUMNS if column in CANDIDATE_PREDICTORS]
if leakage_used:
    raise ValueError(f'Leakage columns cannot be used as predictors: {leakage_used}')

print(f'Data path: {DATA_PATH}')
print(f'Raw dataset shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
print(f'Candidate predictors: {len(CANDIDATE_PREDICTORS)}')

## Candidate Predictor Rationale

The candidate pool is compact and explainable. It combines audience reaction, critic reaction, movie context, and early review behavior while excluding direct revenue leakage.

In [ ]:
predictor_rationale = pd.DataFrame([
    ('audienceScore', 'Audience reaction', 'Direct audience evaluation of the movie'),
    ('tomatoMeter', 'Critic reaction', 'Professional critic score'),
    ('runtimeMinutes', 'Movie context', 'Runtime can proxy format and release type'),
    ('initial_top_critic_review_count', 'Early attention', 'Volume of top-critic attention in the first review window'),
    ('initial_positive_review_ratio', 'Early sentiment', 'Share of early reviews that are positive'),
    ('initial_combined_sentiment_score', 'Early sentiment', 'Weighted early-review sentiment score'),
    ('log_initial_review_count', 'Early attention', 'Logged early review volume'),
], columns=['predictor', 'role', 'reason'])

display(predictor_rationale)

## Data Preparation

Rows are retained only when the target and all candidate predictors are available. This keeps model comparisons fair because each model is evaluated on the same observations.

In [ ]:
predictive_df = df[REQUIRED_COLUMNS].copy()
for column in ['box_office_num', TARGET, *CANDIDATE_PREDICTORS]:
    predictive_df[column] = pd.to_numeric(predictive_df[column], errors='coerce')

predictive_df = predictive_df.dropna(subset=[TARGET, *CANDIDATE_PREDICTORS]).copy()
predictive_df = predictive_df[np.isfinite(predictive_df[TARGET])].copy()

quality_summary = pd.DataFrame({
    'non_null_count': predictive_df[[TARGET, *CANDIDATE_PREDICTORS]].notna().sum(),
    'mean': predictive_df[[TARGET, *CANDIDATE_PREDICTORS]].mean(),
    'std': predictive_df[[TARGET, *CANDIDATE_PREDICTORS]].std(),
})

print(f'Rows retained for predictive modelling: {predictive_df.shape[0]:,}')
display(quality_summary)
display(predictive_df[[TARGET, *CANDIDATE_PREDICTORS]].describe().T)

## Train/Test Split and Evaluation Functions

The held-out test set is used only for final evaluation. Cross-validation is performed on the training set only.

In [ ]:
X = predictive_df[CANDIDATE_PREDICTORS].copy()
y = predictive_df[TARGET].copy()

X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
    X,
    y,
    predictive_df.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

train_df = predictive_df.loc[train_idx].copy()
test_df = predictive_df.loc[test_idx].copy()
cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

split_summary = pd.DataFrame([
    {'set': 'train', 'rows': len(train_df), 'pct': len(train_df) / len(predictive_df) * 100},
    {'set': 'test', 'rows': len(test_df), 'pct': len(test_df) / len(predictive_df) * 100},
])
display(split_summary)


def evaluate_predictions(y_true: pd.Series, y_pred: np.ndarray) -> dict:
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    return {'rmse': rmse, 'mae': mae, 'r2': r2}


def cv_rmse_for_columns(columns: list[str]) -> float:
    scores = cross_val_score(
        LinearRegression(),
        X_train[columns],
        y_train,
        cv=cv,
        scoring='neg_root_mean_squared_error',
    )
    return float(-scores.mean())


def fit_statsmodels_ols(train_frame: pd.DataFrame, columns: list[str]):
    train_X = sm.add_constant(train_frame[columns], has_constant='add')
    return sm.OLS(train_frame[TARGET], train_X).fit()


def evaluate_linear_model(name: str, columns: list[str]) -> tuple[dict, object, np.ndarray, np.ndarray]:
    fitted = fit_statsmodels_ols(train_df, columns)
    train_pred = fitted.predict(sm.add_constant(train_df[columns], has_constant='add'))
    test_pred = fitted.predict(sm.add_constant(test_df[columns], has_constant='add'))
    train_metrics = evaluate_predictions(train_df[TARGET], train_pred)
    test_metrics = evaluate_predictions(test_df[TARGET], test_pred)
    result = {
        'model': name,
        'n_features': len(columns),
        'features': ', '.join(columns),
        'adj_r_squared': float(fitted.rsquared_adj),
        'aic': float(fitted.aic),
        'bic': float(fitted.bic),
        'cv_rmse': cv_rmse_for_columns(columns),
        'train_rmse': train_metrics['rmse'],
        'train_mae': train_metrics['mae'],
        'train_r2': train_metrics['r2'],
        'test_rmse': test_metrics['rmse'],
        'test_mae': test_metrics['mae'],
        'test_r2': test_metrics['r2'],
    }
    return result, fitted, np.asarray(train_pred), np.asarray(test_pred)

## Baseline and Candidate Models

The first comparison checks whether the selected predictors improve meaningfully over a mean-only benchmark and whether the inferential model remains competitive for prediction.

In [ ]:
model_results = []
model_objects = {}
prediction_store = {}

baseline_mean = float(y_train.mean())
baseline_train_pred = np.full(len(y_train), baseline_mean)
baseline_test_pred = np.full(len(y_test), baseline_mean)
baseline_train_metrics = evaluate_predictions(y_train, baseline_train_pred)
baseline_test_metrics = evaluate_predictions(y_test, baseline_test_pred)
baseline_result = {
    'model': 'Baseline 0: Mean-only predictor',
    'n_features': 0,
    'features': 'intercept only',
    'adj_r_squared': np.nan,
    'aic': np.nan,
    'bic': np.nan,
    'cv_rmse': baseline_train_metrics['rmse'],
    'train_rmse': baseline_train_metrics['rmse'],
    'train_mae': baseline_train_metrics['mae'],
    'train_r2': baseline_train_metrics['r2'],
    'test_rmse': baseline_test_metrics['rmse'],
    'test_mae': baseline_test_metrics['mae'],
    'test_r2': baseline_test_metrics['r2'],
}
model_results.append(baseline_result)
prediction_store[baseline_result['model']] = {'train_pred': baseline_train_pred, 'test_pred': baseline_test_pred, 'features': []}

for name, columns in [
    ('Model 1: Inferential three-variable regression', INFERENTIAL_MODEL_PREDICTORS),
    ('Model 2: Full candidate-pool regression', CANDIDATE_PREDICTORS),
]:
    result, fitted, train_pred, test_pred = evaluate_linear_model(name, columns)
    model_results.append(result)
    model_objects[name] = fitted
    prediction_store[name] = {'train_pred': train_pred, 'test_pred': test_pred, 'features': columns}

baseline_models_table = pd.DataFrame(model_results)
display(baseline_models_table)

## Explainable Feature Selection

Two transparent selection methods are used:

- **Best-subset selection** checks every possible predictor combination.
- **Forward-stepwise selection** adds one predictor at a time.

The primary selection criterion is lowest cross-validated RMSE. BIC and number of predictors are tie-breakers to keep the final model compact.

In [ ]:
subset_records = []
for size in range(1, len(CANDIDATE_PREDICTORS) + 1):
    for subset in itertools.combinations(CANDIDATE_PREDICTORS, size):
        columns = list(subset)
        fit = fit_statsmodels_ols(train_df, columns)
        subset_records.append({
            'method': 'Best subset',
            'n_features': size,
            'features': columns,
            'adj_r_squared': float(fit.rsquared_adj),
            'aic': float(fit.aic),
            'bic': float(fit.bic),
            'cv_rmse': cv_rmse_for_columns(columns),
        })

best_subset_table = pd.DataFrame(subset_records)
best_subset_winners = (
    best_subset_table
    .sort_values(['cv_rmse', 'bic', 'n_features'])
    .groupby('n_features', as_index=False)
    .first()
    .sort_values('n_features')
)

remaining = CANDIDATE_PREDICTORS.copy()
selected = []
forward_records = []
while remaining:
    candidate_step_results = []
    for feature in remaining:
        trial_features = selected + [feature]
        fit = fit_statsmodels_ols(train_df, trial_features)
        candidate_step_results.append({
            'step': len(trial_features),
            'features': trial_features.copy(),
            'added_feature': feature,
            'adj_r_squared': float(fit.rsquared_adj),
            'aic': float(fit.aic),
            'bic': float(fit.bic),
            'cv_rmse': cv_rmse_for_columns(trial_features),
        })
    step_df = pd.DataFrame(candidate_step_results).sort_values(['cv_rmse', 'bic', 'step'])
    best_step = step_df.iloc[0].to_dict()
    forward_records.append(best_step)
    selected = best_step['features']
    remaining.remove(best_step['added_feature'])

forward_stepwise_table = pd.DataFrame(forward_records)

display(best_subset_winners)
display(forward_stepwise_table)
print(f'Total best-subset models evaluated: {len(best_subset_table):,}')

## Final Model Comparison and Selection

The final selected model is chosen from the benchmark, inferential, full, best-subset, and forward-stepwise models. When best-subset and forward-stepwise are equivalent, the best-subset model is reported because it comes from exhaustive search.

In [ ]:
best_subset_choice = best_subset_table.sort_values(['cv_rmse', 'bic', 'n_features']).iloc[0]
forward_choice = forward_stepwise_table.sort_values(['cv_rmse', 'bic', 'step']).iloc[0]

best_subset_features = list(best_subset_choice['features'])
forward_features = list(forward_choice['features'])

for name, columns in [
    ('Model 3: Best-subset selected regression', best_subset_features),
    ('Model 4: Forward-stepwise selected regression', forward_features),
]:
    result, fitted, train_pred, test_pred = evaluate_linear_model(name, columns)
    model_results.append(result)
    model_objects[name] = fitted
    prediction_store[name] = {'train_pred': train_pred, 'test_pred': test_pred, 'features': columns}

model_comparison_table = pd.DataFrame(model_results)
selection_ready_table = model_comparison_table.copy()
selection_ready_table['bic_for_sort'] = selection_ready_table['bic'].fillna(np.inf)
selection_ready_table = selection_ready_table.sort_values(
    ['cv_rmse', 'test_rmse', 'bic_for_sort', 'n_features'],
    ascending=[True, True, True, True],
)

final_model_name = 'Model 3: Best-subset selected regression'
final_model_features = prediction_store[final_model_name]['features']
final_model_fit = model_objects[final_model_name]
final_row = model_comparison_table.loc[model_comparison_table['model'] == final_model_name].iloc[0]

best_subset_row = model_comparison_table.loc[model_comparison_table['model'] == 'Model 3: Best-subset selected regression'].iloc[0]
forward_row = model_comparison_table.loc[model_comparison_table['model'] == 'Model 4: Forward-stepwise selected regression'].iloc[0]
models_tie = np.isclose(best_subset_row['cv_rmse'], forward_row['cv_rmse']) and np.isclose(best_subset_row['test_rmse'], forward_row['test_rmse'])

selection_commentary = pd.DataFrame([
    {'category': 'Primary criterion', 'decision': 'Lowest cross-validated RMSE on the training set'},
    {'category': 'Best-subset result', 'decision': f"{best_subset_row['model']} with CV RMSE {best_subset_row['cv_rmse']:.4f}"},
    {'category': 'Forward-stepwise check', 'decision': 'Ties the best-subset model with the same features and performance' if models_tie else 'Does not beat the best-subset model'},
    {'category': 'Final selection rule', 'decision': 'Report the best-subset model when tied, because it comes from exhaustive search'},
])

display(model_comparison_table.sort_values(['cv_rmse', 'test_rmse', 'n_features']))
display(selection_commentary)
print(f'Final selected model: {final_model_name}')
print(f'Final selected features: {final_model_features}')
print(f"Test RMSE: {final_row['test_rmse']:.4f}")
print(f"Test MAE: {final_row['test_mae']:.4f}")
print(f"Test R-squared: {final_row['test_r2']:.4f}")

## Final Model Diagnostics

Prediction diagnostics show where the selected model performs well and where large errors remain.

In [ ]:
final_test_pred = prediction_store[final_model_name]['test_pred']
final_train_pred = prediction_store[final_model_name]['train_pred']

diagnostic_frame = test_df[['id', 'title', 'box_office_num', TARGET]].copy()
diagnostic_frame['predicted_log_box_office'] = final_test_pred
diagnostic_frame['residual'] = diagnostic_frame[TARGET] - diagnostic_frame['predicted_log_box_office']
diagnostic_frame['absolute_error'] = diagnostic_frame['residual'].abs()
diagnostic_frame['predicted_box_office_num'] = np.power(10, diagnostic_frame['predicted_log_box_office'])
diagnostic_frame['box_office_error'] = diagnostic_frame['box_office_num'] - diagnostic_frame['predicted_box_office_num']

largest_error_table = diagnostic_frame.sort_values('absolute_error', ascending=False).head(15)
display(largest_error_table)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(diagnostic_frame[TARGET], diagnostic_frame['predicted_log_box_office'], alpha=0.35, s=25)
line_min = min(diagnostic_frame[TARGET].min(), diagnostic_frame['predicted_log_box_office'].min())
line_max = max(diagnostic_frame[TARGET].max(), diagnostic_frame['predicted_log_box_office'].max())
axes[0].plot([line_min, line_max], [line_min, line_max], color='red', linewidth=1)
axes[0].set_title('Actual vs predicted')
axes[0].set_xlabel('Actual log_box_office')
axes[0].set_ylabel('Predicted log_box_office')

axes[1].scatter(diagnostic_frame['predicted_log_box_office'], diagnostic_frame['residual'], alpha=0.35, s=25)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title('Residuals vs predicted')
axes[1].set_xlabel('Predicted log_box_office')
axes[1].set_ylabel('Residual')

sns.histplot(diagnostic_frame['residual'], kde=True, ax=axes[2], color='#457b9d')
axes[2].set_title('Prediction error distribution')
axes[2].set_xlabel('Residual')
plt.tight_layout()
plt.show()

## Feature Contribution

Standardized coefficients are used to compare feature importance on a common scale. This is still an explainable linear-regression interpretation, not a black-box importance measure.

In [ ]:
standardized_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression()),
])
standardized_pipeline.fit(train_df[final_model_features], train_df[TARGET])
standardized_coefs = standardized_pipeline.named_steps['regressor'].coef_
contribution_table = pd.DataFrame({
    'feature': final_model_features,
    'standardized_coef': standardized_coefs,
    'absolute_standardized_coef': np.abs(standardized_coefs),
}).sort_values('absolute_standardized_coef', ascending=False)

display(contribution_table)

## Predictive Conclusion

The final selected model is the 5-feature best-subset regression. It balances predictive accuracy and interpretability while avoiding direct revenue leakage and avoiding the removed interaction feature.

In [ ]:
print('Predictive conclusion')
print(f'Selected model: {final_model_name}')
print(f'Number of predictors: {int(final_row["n_features"])}')
print(f'Cross-validated RMSE: {final_row["cv_rmse"]:.4f}')
print(f'Test RMSE: {final_row["test_rmse"]:.4f}')
print(f'Test MAE: {final_row["test_mae"]:.4f}')
print(f'Test R-squared: {final_row["test_r2"]:.4f}')
print('Interpretation: the model explains a meaningful share of unseen variation in log_box_office using a compact, explainable feature set.')